In [23]:
"""
SFT Demo 脚本
"""
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B-Base/")

In [29]:
# 训练过程，message_list当中，包含user和assistant的消息
message_list = [
    {"role":"user","content":"你好"},
    {"role":"assistant","content":"你好呀！"},
    {"role":"user","content":"早上好"},
    {"role":"assistant","content":"hello呀"},
]

message_list2 = [
     {"role":"user","content":"你好，G老师"},
    {"role":"assistant","content":"你好呀！，今天有什么帮你？"},
]

# 实际使用时，会通过tokenizer.apply_chat_template进行分词，得到input_ids
res = tokenizer.apply_chat_template(message_list,tokenize=False)
# attention_mask:
use_attention_mask_res = tokenizer.apply_chat_template([message_list,message_list2],padding=True)

print(res)

<|im_start|>user
你好<|im_end|>
<|im_start|>assistant
你好呀！<|im_end|>
<|im_start|>user
早上好<|im_end|>
<|im_start|>assistant
<think>

</think>

hello呀<|im_end|>



In [19]:
# 训练过程，message_list当中，包含user和assistant的消息
message_list = [
    {"role":"user","content":"你好"},
    {"role":"assistant","content":"你好呀！"},
    {"role":"user","content":"早上好！"}
]

inference_res = tokenizer.apply_chat_template(message_list,tokenize=False,add_generation_prompt=True,enable_thinking = True)
print(inference_res)

<|im_start|>user
你好<|im_end|>
<|im_start|>assistant
你好呀！<|im_end|>
<|im_start|>user
早上好！<|im_end|>
<|im_start|>assistant



In [34]:

from datasets import load_dataset
from typing import Dict
def get_train_data():
    """
    加载训练数据
    """
    dataset = load_dataset("data/ultrachat_200k")
    train_dataset = dataset["train_sft"]
    # 遍历其中的数据，对每条数据，使用tokenizer.apply_chat_template进行编码
    result_list = []
    train_data_size = 200

    for i in range(train_data_size):
        message_list:list[Dict] = train_dataset[i]["messages"]
        message_list.insert(0,{"role":"system","content":"You are a helpful assistant."})
        result_dict = tokenizer.apply_chat_template(message_list,)
        result_list.append(result_dict)

    return result_list


In [35]:
from transformers import PreTrainedTokenizerFast
import torch
from typing import List

def create_answer_mask(input_ids,tokenizer:PreTrainedTokenizerFast):
    """
    创建answer mask，从input_ids当中找出assistant回答的部分，然后输出一个与input_ids相同shape的mask，
    后续将其与pad_mask进行逻辑与操作，得到最终的mask，用以计算损失
    """

    
    # 构建answer mask，输入的input_ids为批量 tokenize之后的数据，对于每一条数据，查找当中assistant回答的部分，将其设置为1

    # 1. 构造一个和input_ids相同shape的全0矩阵
    answer_mask = torch.zeros_like(input_ids)

    # 2. 遍历input_ids中的每一条数据，查找assistant回答的部分，将其设置为1
    eos_token_id = tokenizer.encode('<|im_end|>')[0]
    for idx,ids in enumerate(input_ids):
        # 获取到所有的eos_position
        eos_position:List = torch.where(ids == eos_token_id)[0].tolist()

        # 排除第一个eos_position: 第一个对应的是system prompt
        eos_position = eos_position[1:]
        # 解析获得user_ends和assistant_ends
        user_ends,assistant_ends = _parse_conversation_turns(eos_position)
        # 设置answer mask
        _set_answer_masks(answer_mask[idx],user_ends,assistant_ends)   
    
    # 结果返回:
    return answer_mask

def _parse_conversation_turns(eos_positions:List[int]):
    """
    输入eos_positions，输出user所对应的end位置和assistant所对应的end位置。

    以下面的对话为例：
    <|im_start|>system
    You are a helpful assistant.<|im_end|>
    <|im_start|>user
    什么是习惯？<|im_end|>
    <|im_start|>assistant
    习惯是指在一定时间内重复执行的行为。<|im_end|>
    <|im_start|>user
    如何培养一个习惯<|im_end|>
    <|im_start|>assistant
    21天培养法，每天坚持xxx<|im_end|>

    假设第一个eos_token_id index为5，第二个为10，第三个为15，第四个为20，第五个为25，
    那么输入的eos_token_id为：[10,15,20,25]
    user_turns为从第一个开始取（具体索引位置需要加一，因为eos_token_id后面还有一个\n换行符），每隔一个取一次，assistant_turns为从第二个开始取，每隔一个取一次。

    输出结果为：
        user_turns:[11,21]
        assistant_ends:[16,26]
    """

    use_ends = [pos+1 for pos in eos_positions[::2]]
    assistant_ends = [pos+1 for pos in eos_positions[1::2]]

    return use_ends,assistant_ends

def _set_answer_masks(mask,user_ends,assistant_ends):
    """
    将mask当中，assistant回答的部分，设置为1（原地修改，不返回新的mask），其余部分保持为0

    以下面的对话为例：
    <|im_start|>system
    You are a helpful assistant.<|im_end|>
    <|im_start|>user
    什么是习惯？<|im_end|>
    <|im_start|>assistant
    习惯是指在一定时间内重复执行的行为。<|im_end|>
    <|im_start|>user
    如何培养一个习惯<|im_end|>
    <|im_start|>assistant
    21天培养法，每天坚持xxx<|im_end|>

    假设第一个eos_token_id index为5，第二个为10，第三个为15，第四个为20，第五个为25，
    那么user_turns:[11,21]，assistant_ends:[16,26]

    user_ends当中的索引指向的是<|im_end|>之后的\n的索引，
    assistant_ends当中的索引指向的是<|im_end|>之后的\n的索引，
    要想获取到assistant的回答的起始位置，就需要再跳过\n,<|im_start|>,assistant 这三个token，所以需要加3.
    要想获取到assistant的回答的结束位置，就需要往前跳一个<|im_end|>，所以需要减1.
    """
    num_user_turns = len(user_ends)
    num_assistant_turns = len(assistant_ends)
    if num_user_turns == num_assistant_turns:
        for user_end,assistant_end in zip(user_ends,assistant_ends):
            answer_start = user_end + 3
            answer_end = assistant_end - 1
            mask[answer_start:answer_end] = 1

    elif num_user_turns == num_assistant_turns + 1:
        for user_end,assistant_end in zip(user_ends[:-1],assistant_ends):
            answer_start = user_end + 3
            answer_end = assistant_end - 1
            mask[answer_start:answer_end] = 1
        
        # 处理最后一轮被截断的助手回答
        last_user_end = user_ends[-1] 
        last_answer_start = last_user_end + 3
        mask[last_answer_start:] = 1


In [49]:

data = get_train_data()
message_list = [
    {"role":"system","content":"你是一个智能助手"},
    {"role":"user","content":"你好"},
    {"role":"assistant","content":"你好呀！"},
    {"role":"user","content":"早上好"},
    {"role":"assistant","content":"hello呀"},
]
input_ids= tokenizer.apply_chat_template(message_list,tokenize=True)["input_ids"]

input_ids = torch.tensor([input_ids],dtype=torch.long)
#input_ids:(seq_len,)
# input_ids: (batch_size, seq_len)
assistant_answer_mask = create_answer_mask(input_ids=input_ids,tokenizer=tokenizer)

In [50]:
assistant_answer_mask

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0]])

In [51]:
input_id_list=input_ids.tolist()[0]
assistant_answer_mask_list = assistant_answer_mask.tolist()[0]
print(input_id_list)



[151644, 8948, 198, 56568, 101909, 100168, 110498, 151645, 198, 151644, 872, 198, 108386, 151645, 198, 151644, 77091, 198, 108386, 104256, 6313, 151645, 198, 151644, 872, 198, 105083, 52801, 151645, 198, 151644, 77091, 198, 151667, 271, 151668, 271, 14990, 104256, 151645, 198]


In [52]:
for input_id,is_assistant_answer in zip(input_id_list,assistant_answer_mask_list):
    print('当前的token_id为：',input_id, "解码之后的token是：",repr(tokenizer.decode(input_id)),"当前token是否为assistant_answer:",is_assistant_answer)

当前的token_id为： 151644 解码之后的token是： '<|im_start|>' 当前token是否为assistant_answer: 0
当前的token_id为： 8948 解码之后的token是： 'system' 当前token是否为assistant_answer: 0
当前的token_id为： 198 解码之后的token是： '\n' 当前token是否为assistant_answer: 0
当前的token_id为： 56568 解码之后的token是： '你' 当前token是否为assistant_answer: 0
当前的token_id为： 101909 解码之后的token是： '是一个' 当前token是否为assistant_answer: 0
当前的token_id为： 100168 解码之后的token是： '智能' 当前token是否为assistant_answer: 0
当前的token_id为： 110498 解码之后的token是： '助手' 当前token是否为assistant_answer: 0
当前的token_id为： 151645 解码之后的token是： '<|im_end|>' 当前token是否为assistant_answer: 0
当前的token_id为： 198 解码之后的token是： '\n' 当前token是否为assistant_answer: 0
当前的token_id为： 151644 解码之后的token是： '<|im_start|>' 当前token是否为assistant_answer: 0
当前的token_id为： 872 解码之后的token是： 'user' 当前token是否为assistant_answer: 0
当前的token_id为： 198 解码之后的token是： '\n' 当前token是否为assistant_answer: 0
当前的token_id为： 108386 解码之后的token是： '你好' 当前token是否为assistant_answer: 0
当前的token_id为： 151645 解码之后的token是： '<|im_end|>' 当前token是否为assistant_answer: 0
当前的toke

In [ ]:
# SFT,损失计算函数，具体逻辑：
def compute_losss(output_logits, labels, assistant_answer_mask):
    """
    output_logits: 模型前向传播之后，输出的结果 , shape batch_size, seq_len, vocab_size
    labels: 真实的标签 batch_size, seq_len
    assistant_answer_mask: 模型回答掩码
    """

    # 1、需要对output_logits进行log softmax，得到对数概率
    # log_probs: batch_size, seq_len, vocab_size
    log_probs = torch.log_softmax(output_logits, dim=-1)

    # 2、找到模型输出答案所对应的token的概率是多少
    result = torch.gather(
        log_probs,
        dim=-1,
        index=labels
    )

    result = result.squeeze(-1) 

    # negative_log_probs.shape [batch_size,seq_len]
    negative_log_probs = result * (-1)

    # 哈达玛积，对应位置相乘，需要算损失的token，乘以1，不需要算损失的token，乘以0，
    masked_negative_log_probs = negative_log_probs * assistant_answer_mask

    # 将序列当中所有token的对数概率加起来，除以总的token数量
    average_loss  = masked_negative_log_probs.sum() / assistant_answer_mask.sum()

    return average_loss



    


In [ ]:
# torch.gather：找到一个张量当中，对应索引位置的值是多少
# 假设vocab_size=3
log_probs = torch.tensor(
    [ [[0.1, 0.3, 0.6],[0.2, 0.5, 0.3],[0.4,0.4,0.2]] ]
)
labels = torch.tensor(
    [[2,0,1]]
)
result = torch.gather(
    log_probs,
    dim=-1,
    index=labels.unsqueeze(-1)
)
result = result.squeeze(-1)


In [58]:
result

tensor([[[0.6000],
         [0.2000],
         [0.4000]]])